### Pydantic 모델
- 정의된 데이터만 전송되도록 요청 바디 검증

In [1]:
from datetime import datetime

from pydantic import BaseModel, PositiveInt, ValidationError


class User(BaseModel):
    id: int  
    name: str = 'John Doe'  
    signup_ts: datetime | None  
    tastes: dict[str, PositiveInt]  


external_data = {
    'id': 123,
    'signup_ts': '2019-06-01 12:22',  
    'tastes': {
        'wine': 9,
        b'cheese': 7,  
        'cabbage': '1',  
    },
}

# 객체 생성시
#  user = USER(id=1234, signup_ts = "2019-06-01 12:22")
user = User(**external_data)  

print(user.id)  
#> 123
print('----------------')
print(user.model_dump())  # 모델의 필드와 값을 dict 형태로 반환


123
----------------
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [2]:

external_data = {'id': 'not an int', 'tastes': {}}  
try:
    User(**external_data)  
except ValidationError as e:
    print(e.errors())


[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [9]:
# Fruit : name - 문자열, color - red or green, weight - 0보다 크고 3.5같은 float 타입, bazam-dict[키-str, [tuple(int, bool, float)]]

from typing import Annotated, Literal
from annotated_types import Gt

class Fruit(BaseModel):
    name:str
    color: Literal["red", "green"]
    weight: Annotated[float, Gt(0)]
    bazam:dict[str, list[tuple[int, bool, float]]]

In [11]:
fruit = Fruit(name='Apple', color='red', weight=4.2, bazam={'foobar':[(3, True, 3.14)]})
print(fruit)

name='Apple' color='red' weight=4.2 bazam={'foobar': [(3, True, 3.14)]}


In [ ]:
fruit = Fruit(name='Apple', color='red', bazam={'foobar':[(3, True, 3.14)]})

In [17]:
# Book - code(4자리), category(소설, 과학, 역사, 경제, 자기계발, 컴퓨터), title - 문자, writer - 문자, price - int이면서 0보다 크다

from pydantic import StringConstraints

class Book(BaseModel):
    code:Annotated[str, StringConstraints(min_length=4, max_length=4)]
    category : Literal["소설", "과학", "역사", "경제", "자기계발", "컴퓨터"]
    title:str
    writer:str
    price:Annotated[int, Gt(0)]

Harry_potter = Book(code = "1234", category="소설", title="Harry Potter and the Sorcerer's Stone", writer="J. K. Rowling", price=20000)
print(Harry_potter)

code='1234' category='소설' title="Harry Potter and the Sorcerer's Stone" writer='J. K. Rowling' price=20000


In [19]:
try:
    Book(code='1001', category="경제", price=20000)
except ValidationError as e:
    print(e.errors())

[{'type': 'missing', 'loc': ('title',), 'msg': 'Field required', 'input': {'code': '1001', 'category': '경제', 'price': 20000}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('writer',), 'msg': 'Field required', 'input': {'code': '1001', 'category': '경제', 'price': 20000}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [21]:
# User-id:int, name:문자, age : 14~100, email, role='user' or 'admin'

from pydantic import Field

class User(BaseModel):
    id:int
    name:str
    age:Annotated[int, Field(ge=14, le=100)]
    email:str
    role:Literal["user", "admin"]

In [22]:
try:
    User(id=1, name="홍길동", age=101, email="hong@gmail.com", role='user')
except ValidationError as e:
    print(e.errors())

[{'type': 'less_than_equal', 'loc': ('age',), 'msg': 'Input should be less than or equal to 100', 'input': 101, 'ctx': {'le': 100}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}]


In [28]:
# product : code - 문자, name-문자, category-(food, clothes, electronics), price-0보다 큰 정수, stock- 0이상 정수

class Product(BaseModel):
    code:str
    name:str
    category:Literal["food", "clothes", "electronics"]
    price:Annotated[int, Gt(0)]
    stock:Annotated[int, Field(ge=0)]

In [29]:
try:
    Product(code='11111', name="YOO", category="food", price=1, stock=10)
except ValidationError as e:
    print(e.errors())

In [31]:
class Item(BaseModel):
    item:str
    status:str

class Cart(BaseModel):
    id:int
    item:Item

In [32]:
try:
    cart = Cart(id=1, item=Item(item='Hello-Kitty', status="Happy"))
except ValidationError as e:
    print(e.errors())

In [33]:
class OrderItem(BaseModel):
    product_name:str
    quantity:Annotated[int, Gt(0)]
    price:Annotated[int, Gt(0)]

class Order(BaseModel):
    order_id:int
    customer:str
    items:list[OrderItem]

In [35]:
order = Order(order_id = 1001, customer="나", items=[
    {"product_name":"노트북", "quantity":10, "price":1000011},
    {"product_name":"마우스", "quantity":10, "price":10000},
])

In [41]:
# Comment: writer-문자, content-문자
# Post : id-숫자, title-문자, writer-문자, content-문자, comments: Comment가 여러개

class Comment(BaseModel):
    writer:str
    content:str

class Post(BaseModel):
    id:int
    title:str
    writer:str
    content:str
    comments:list[Comment]

In [43]:
post = Post(id=11, title="sonic", writer="me", content="123456", comments = [{"writer":"meme", "content":"mememememememe"}])